# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riteshy1526/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

My baseline rule ranks content pages that are likely to benefit from a content refresh.

The rule combines four observable signals:

- Days since last update
- Content age
- Traffic trend
- Click-through rate (CTR)

Older pages with declining traffic and low CTR receive higher scores.

This rule provides a simple baseline that will later be compared against a machine learning model.

### Reason Codes

- STALE_CONTENT
- DECLINING_TRAFFIC
- LOW_CTR
- MONITOR

In [23]:
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parents[1]

df = pd.read_csv(
    project_root / "data" / "raw" / "content_refresh_anonymized.csv"
)

df["update_score"] = df["days_since_last_update"].rank(pct=True)
df["age_score"] = df["content_age_days"].rank(pct=True)
df["trend_score"] = (-df["trend_pct"]).rank(pct=True)
df["ctr_score"] = 1 - df["ctr"].rank(pct=True)

df["baseline_score"] = (
    0.40 * df["update_score"]
    + 0.30 * df["age_score"]
    + 0.20 * df["trend_score"]
    + 0.10 * df["ctr_score"]
)

df[["content_id", "baseline_score"]].head()

,content_id,baseline_score
0,content_304f48230142,0.382521
1,content_a1fb4e703a9e,0.716598
2,content_9aa793d4d895,0.416024
3,content_331d6c4de07b,0.587427
4,content_d99b7a2d90ca,0.356531


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Build the Ranked Queue

The baseline score is calculated for every content page.

Pages with higher scores receive higher priority for refresh.

Each page receives:

- Baseline Score
- Reason Code
- Action Label

The ranked queue is saved as:

`work/outputs/baseline_action_score.csv`

In [24]:
# Reason Codes
conditions = [
    df["days_since_last_update"] > 365,
    df["trend_pct"] < -20,
    df["ctr"] < df["ctr"].median()
]

choices = [
    "STALE_CONTENT",
    "DECLINING_TRAFFIC",
    "LOW_CTR"
]

df["reason_code"] = np.select(
    conditions,
    choices,
    default="MONITOR"
)

# Action label
threshold = df["baseline_score"].quantile(0.80)

df["action_label"] = np.where(
    df["baseline_score"] >= threshold,
    "REFRESH_NOW",
    "MONITOR"
)

# Rank
df = df.sort_values(
    by="baseline_score",
    ascending=False
)

# Save output
output_path = project_root / "work" / "outputs" / "baseline_action_score.csv"

output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

print("CSV Saved:")
print(output_path)

df.head()

CSV Saved:
c:\Users\AJIT KUMAR YADAV\OneDrive\Desktop\code\internship-local-project\flyrank-internship-ml\work\outputs\baseline_action_score.csv


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,position_tier,trend_direction,trend_pct,update_score,age_score,trend_score,ctr_score,baseline_score,reason_code,action_label
29384,content_f6fdf87348f6,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,page_3_5,down,-100.0,0.999967,0.788183,0.973809,0.779783,0.909182,STALE_CONTENT,REFRESH_NOW
24216,content_1b4ec72dafd4,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,page_1,down,-100.0,0.999883,0.788083,0.973809,0.779783,0.909118,STALE_CONTENT,REFRESH_NOW
8407,content_d1e915d03c28,client_4e07408562,140.0,0.82,HIGH,2.09,keyword article,informational,NaN,NaN,...,page_3_5,down,-100.0,0.843200,0.987917,0.973809,0.779783,0.906395,DECLINING_TRAFFIC,REFRESH_NOW
26242,content_55a5b1c46474,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,page_1,down,-88.5,0.999967,0.788717,0.914456,0.779783,0.897471,STALE_CONTENT,REFRESH_NOW
786,content_b06198ab8de1,client_9f14025af0,10.0,0.12,LOW,4.88,keyword article,informational,1064.0,7039.0,...,page_3_5,down,-100.0,0.993567,0.754833,0.973809,0.779783,0.896617,DECLINING_TRAFFIC,REFRESH_NOW


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The highest-ranked twenty pages were reviewed manually.

For each page I recorded:

- Recommended action
- Reason code
- Confidence note
- What could make the recommendation incorrect

This review is intended as a sanity check rather than a final business decision.

In [25]:
top20 = df.head(20).copy()

top20["confidence_note"] = "Medium"

top20["what_would_make_it_wrong"] = (
    "Performance changes may be caused by seasonality, temporary ranking fluctuations, or external events."
)

review = top20[
    [
        "content_id",
        "baseline_score",
        "action_label",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

print(review)

review

                 content_id  baseline_score action_label        reason_code  \
29384  content_f6fdf87348f6        0.909182  REFRESH_NOW      STALE_CONTENT   
24216  content_1b4ec72dafd4        0.909118  REFRESH_NOW      STALE_CONTENT   
8407   content_d1e915d03c28        0.906395  REFRESH_NOW  DECLINING_TRAFFIC   
26242  content_55a5b1c46474        0.897471  REFRESH_NOW      STALE_CONTENT   
786    content_b06198ab8de1        0.896617  REFRESH_NOW  DECLINING_TRAFFIC   
23618  content_4ed2bf493735        0.896617  REFRESH_NOW  DECLINING_TRAFFIC   
18744  content_5f5b48d17daa        0.896617  REFRESH_NOW  DECLINING_TRAFFIC   
11722  content_6efb8fa48ebe        0.896617  REFRESH_NOW  DECLINING_TRAFFIC   
22926  content_c89743570f09        0.896617  REFRESH_NOW  DECLINING_TRAFFIC   
4666   content_d6d5bc71c047        0.896617  REFRESH_NOW  DECLINING_TRAFFIC   
28057  content_b9cc4ac6de81        0.896617  REFRESH_NOW  DECLINING_TRAFFIC   
9417   content_c3dd69918c8c        0.896617  REFRESH

,content_id,baseline_score,action_label,reason_code,confidence_note,what_would_make_it_wrong
29384,content_f6fdf87348f6,0.909182,REFRESH_NOW,STALE_CONTENT,Medium,Performance changes may be caused by seasonali...
24216,content_1b4ec72dafd4,0.909118,REFRESH_NOW,STALE_CONTENT,Medium,Performance changes may be caused by seasonali...
8407,content_d1e915d03c28,0.906395,REFRESH_NOW,DECLINING_TRAFFIC,Medium,Performance changes may be caused by seasonali...
26242,content_55a5b1c46474,0.897471,REFRESH_NOW,STALE_CONTENT,Medium,Performance changes may be caused by seasonali...
786,content_b06198ab8de1,0.896617,REFRESH_NOW,DECLINING_TRAFFIC,Medium,Performance changes may be caused by seasonali...
23618,content_4ed2bf493735,0.896617,REFRESH_NOW,DECLINING_TRAFFIC,Medium,Performance changes may be caused by seasonali...
18744,content_5f5b48d17daa,0.896617,REFRESH_NOW,DECLINING_TRAFFIC,Medium,Performance changes may be caused by seasonali...
11722,content_6efb8fa48ebe,0.896617,REFRESH_NOW,DECLINING_TRAFFIC,Medium,Performance changes may be caused by seasonali...
22926,content_c89743570f09,0.896617,REFRESH_NOW,DECLINING_TRAFFIC,Medium,Performance changes may be caused by seasonali...
4666,content_d6d5bc71c047,0.896617,REFRESH_NOW,DECLINING_TRAFFIC,Medium,Performance changes may be caused by seasonali...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

##  Weak Picks and Leakage Check

Some recommendations may be incorrect because traffic changes are not always caused by content quality.

Possible reasons include:

- Seasonal traffic
- Temporary ranking changes
- External events
- Recent algorithm updates

The baseline rule excludes future-window metrics and identifier fields to reduce the risk of data leakage.

The scoring rule uses only information that would reasonably be available before making the recommendation.

In [26]:
excluded_columns = [
    "content_id",
    "client_id",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

print("Excluded Columns")

for col in excluded_columns:
    print("-", col)

print("\nLeakage Check Complete")

print("No excluded columns were used when calculating the baseline score.")

Excluded Columns
- content_id
- client_id
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d

Leakage Check Complete
No excluded columns were used when calculating the baseline score.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.